In [1]:
import yaml

with open("../configs/pretrain_project/silica/baselines/config_graphormer_direct_force.yml") as f:
    config = yaml.safe_load(f)

In [2]:
from matdeeplearn.trainers.base_trainer import BaseTrainer

dataset = BaseTrainer._load_dataset(config["dataset"], config["task"]["run_mode"]) if "src" in config["dataset"] else None
model1 = BaseTrainer._load_model(config["model"], config["dataset"]["preprocess_params"], dataset, 1, 0)[0]
model2 = BaseTrainer._load_model(config["model"], config["dataset"]["preprocess_params"], dataset, 1, 0)[0]
sampler = BaseTrainer._load_sampler(config["optim"], dataset, 1, 0) if "src" in config["dataset"] else None

/net/csefiles/coc-fung-cluster/Qianyu/stable_md/MatDeepLearn_dev/matdeeplearn/preprocessor/datasets.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slic

In [3]:
import torch

checkpoint_pth = "/net/csefiles/coc-fung-cluster/Qianyu/stable_md/MatDeepLearn_dev/results/2024-10-08-23-10-15-016-graphormer3d_direct_force_6layer/checkpoint_0/best_checkpoint.pt"
model1.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])
model2.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])

/tmp/ipykernel_3213300/3520616574.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])
/t

<All keys matched successfully>

In [4]:
from torch_geometric.loader import DataLoader

loader = DataLoader(dataset['test'], batch_size=1, shuffle=False)

In [7]:
from torch.profiler import profile, record_function, ProfilerActivity

model1 = model1.to("cpu")
batch = next(iter(loader)).to("cpu")
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with record_function("model_inference"):
        model1(batch)

In [8]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
              model_inference        20.97%     848.635ms       100.00%        4.047s        4.047s             1  
              softmax_dropout         0.01%     458.315us        25.51%        1.032s     147.484ms             7  
                 <forward op>         0.01%     218.942us        25.49%        1.032s     147.397ms             7  
         aten::native_dropout         0.02%     650.283us        20.27%     820.393ms     117.199ms             7  
                   aten::gelu        20.24%     819.362ms        20.24%     819.362ms     102.420ms             8  
                    aten::mul         9.22%     373.178ms        16.93% 

: 

In [5]:
import numpy as np
import torch

model = model1.to("cuda:1")
batch = next(iter(loader)).to("cuda:1")

starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
repetitions = 300
timings = np.zeros((repetitions,1))

#GPU-WARM-UP
for _ in range(10):
    _ = model(batch)

# MEASURE PERFORMANCE
with torch.no_grad():
    for rep in range(repetitions):
        starter.record()
        _ = model(batch)
        ender.record()
        # WAIT FOR GPU SYNC
        torch.cuda.synchronize()
        curr_time = starter.elapsed_time(ender)
        timings[rep] = curr_time

mean_syn = np.sum(timings) / repetitions
std_syn = np.std(timings)
print(f"Mean inference time: {mean_syn:.2f} ms")
print(f"Std  inference time: {std_syn:.2f} ms")

Mean inference time: 4.51 ms
Std  inference time: 0.09 ms


In [8]:
import torch.nn.functional as F

sum_ = 0
model1 = model1.to("cuda:1")
model2 = model2.to("cuda:2")

for batch in loader:
    result = model1(batch.to("cuda:1"))
    force_q = result["pos_grad"]
    result = model2(batch.to("cuda:2"))
    force_f = result["pos_grad"]
    sum_ += F.l1_loss(force_q.to("cpu"), force_f.to("cpu")).item()
sum_ / len(loader)